# report08 — 드론 RCS·마이크로도플러 결과 (밴드별·자세별)

**핵심.** 탐지는 표적이 레이더 눈에 **얼마나 밝은가(RCS, σ)**에서 출발한다. 이 리포트는 드론 5종의 **절대 RCS** 와 프로펠러 **마이크로도플러 지문**을 자작 SBR+PO 로 산출하고, 그 절대값을 **실측 문헌과 밴드·지표·크기·평균규약을 맞춰** 견준다. **절대 dBsm 판정은 보류한다** — 소형 쿼드의 문헌 절대앵커 자체가 서로 **12 dB 넘게** 어긋나(같은 자세-peak 이 다른 실험실 방위-mean 보다 아래에 오는, 물리적으로 불가능한 산포) 절대 판정의 기준이 될 수 없기 때문이다(§6). 밴드·지표·크기·평균규약을 모두 맞춘 **유일한 사과-대-사과 앵커**(multiband Phantom 3 방위 선형평균)와는 우리 350 mm 정합기가 **0.4~2.9 dB 안**에서 맞는다. 그래서 검출은 σ 밴드로 제시하고 **상대 결론(모드·파형 비교)만** 주장한다.

| 이 리포트의 척추 |  |
|---|---|
| **① Sionna 의 공백** | 스톡 Sionna `PathSolver` 는 표적 σ 를 아예 주지 않고(→report06), 그 위에 얹은 SBR+PO 도 해석해(평판 σ=4πA²/λ²·구 σ=πr²)로는 **방법이 옳음만** 검증될 뿐 — 파이프라인 안에 드론의 **절대 σ 를 대조할 자체 기준이 없다.** 밝기 차이는 크게는 8 dB 에 이른다. |
| **② 선행 연구의 방식** | 소형 멀티로터의 RCS 는 **실측 문헌**이 기준이다. S밴드(3–6 GHz) 실측 aspect-peak 은 Li & Ling 2017(IEEE AWPL): Phantom 2(350 mm) **−27.5** ~ Inspire 1(560 mm) **−13.7 dBsm**(등급 [N] — 워크스페이스 노트 근거, PDF 부재). ⚠ 그 문헌은 **방위평균을 보고하지 않는다** — 'S밴드 방위평균 포락선' 은 aspect-peak 과 자세 스프레드에서 **유도해야 하는 2차 산물**이므로 판정 근거로 쓰지 않는다. 대조는 **peak↔peak** 로만 한다. ⚠ 소형드론 RCS 는 **저주파로 갈수록 떨어진다**(공진/레일리) — 同 문헌에서 3–6 GHz 는 12–15 GHz 보다 평균 ~12 dB 낮다(15 GHz 방위평균: Mavic Pro −17·Phantom 4 −15 dBsm, Ezuma arXiv:1911.05926/2102.11954). |
| **③ 쓴 라이브러리·결합** | 표적 σ 는 **자작 SBR+PO** 로 계산한다(절차 →report07) — Sionna 가 쓰는 **Mitsuba 3 광선엔진을 그대로 재사용**하고 그 위에 PO 표면적분만 얹어(새 광선엔진 없음·중복계산 없음) 복소장 E 를 직접 낸다. BVH SBR+PO(arXiv:2604.09243)와 같은 계열이다(적용범위 차이는 report06 §4). |
| **④ 검증** | 우리 σ 는 **크기 순서·자세 구조·자릿수·대역 추세**를 재현한다. **절대 레벨은 판정 보류다**: 밴드·지표·기하·평균규약을 모두 맞춘 사과-대-사과 앵커(multiband Phantom 3, 방위 **선형평균**)와 우리 350 mm 정합기(phantom4)는 3.5/5.2 GHz 에서 **0.4/0.8 dB**, 1.8 GHz 에서 **2.9 dB** 안에서 맞는다(우리가 약간 어두운 쪽). 같은 측정을 다른 규약으로 요약한 mono3d 와는 3.8~6.1 dB 어긋나는데, 그 3.4 dB 벌어짐 자체가 **선형↔dB영역 평균 규약 차**다(§6). ⚠ 앙각(우리 el=15° ↔ 문헌 el=0°)·편파(우리 스칼라 Γ ↔ 문헌 co-pol)는 아직 정렬되지 않았다. 그래서 검출은 σ 밴드로 제시해 **상대 결론(모드·파형 비교)의 robust 함**을 보인다. |

---


## 📋 이 결과가 어디서 어떻게 나왔나

> 이 절은 **직접 참여하지 않은 사람도 출처를 따라가고 재현할 수 있도록** 넣었습니다. 버전·GPU 는 노트북 생성 시점에 **실제로 읽어온 값**입니다.

### 1️⃣ 무엇을 참고했나

| 항목 | 출처 | 성격 |
|---|---|---|
| 드론 물리 스펙 (대각·무게·프로펠러·로터 수) | DJI 공식 제품 스펙 (Mini 5 Pro · Mavic 4 Pro · Matrice 4E · S1000+ · Phantom 4) | 📄 제조사 스펙 |
| 5종 RCS (3밴드) · 재질 분해 | **`outputs/report2_waveform_rcs.json`** 의 `rcs` / `materials`. `src/viz_report2.py` 가 `src/rcs_sbr.py`(SBR)를 돌려 남긴다 | 🟡 측정 (SBR = Mitsuba 광선 + PO) |
| 호버 rpm 유도 · 블레이드 마이크로도플러 | **`outputs/report1.json`** 의 `articulation`(추력 균형) / `microdoppler`. `src/viz_report3.py` + `src/microdoppler.py` 가 남긴다 | 🟡 측정 (자세별 SBR 산란장) |
| 재질별 반사계수 | **`src/materials.py`** — Sionna RT 와 SBR 이 함께 읽는 단일 진리원 (ITU-R P.2040 기반 + custom) | 📐 물성표 |
| 실측 문헌 드론 RCS (절대값 앵커) | Li & Ling 2017(IEEE AWPL) · Ezuma/Güvenç(arXiv:1911.05926) · Güvenç/NCSU 서베이(arXiv:2402.05909) · Semkin 2020 · Frankford/Björklund(IET RSN) | 📚 실측 문헌 (검증 기준) |

### 2️⃣ 어떤 도구가 무엇을 했나 — **Sionna 내부인가, 우리가 짠 건가**

| 도구 | 하는 일 | 어디서 도는가 |
|---|---|---|
| `sbr` | SBR (`src/rcs_sbr.py`) — **Mitsuba 광선 + PO 표면적분**으로 RCS. 가림(occlusion) 포함 | 🟡 **우리가 짰다** — 다만 광선추적은 Sionna 가 쓰는 **Mitsuba 3 엔진 그대로** (GPU). Sionna 에 RCS 솔버가 없기 때문 |
| `microdoppler` | 마이크로도플러 (`src/microdoppler.py`) — 회전 블레이드의 슬로타임 복소장 → STFT | 🟡 **우리가 짰다** — 자세별 산란장은 SBR(Mitsuba 광선)로 계산 (GPU) |
| `sionna-render` | Sionna RT `Scene.render_to_file()` — 씬·**추적된 광선**·라디오맵을 사진처럼 렌더 | 🟢 **Sionna 내부** (Mitsuba 3 경로추적 렌더러, GPU) |
| `po` | 순수 물리광학 (`src/rcs_po.py`) — 점구름 PO. **가림 없음** | 🔴 **별도** (numpy, CPU). **비교·검증용으로만** 남겨둠 — 기본 엔진은 SBR |
| `matplotlib` | matplotlib — 도표·그래프 | 🔴 **별도** (CPU). 계산 결과를 *그리기만* 한다 |

> 🔑 **이 구분이 이 프로젝트에서 가장 자주 오해받는 지점입니다.**
> - **전파**(경로·지연·도플러·렌더·라디오맵)는 🟢 **Sionna 가** 합니다.
> - **표적 RCS** 는 🟡 우리가 얹은 **PO(물리광학 표면적분)** 가 냅니다 — Sionna 기본 solver 엔 이 산란적분이 없어 경로 이득만 줄 뿐 RCS 를 못 내기 때문입니다. 광선을 쏴 조명면·가림을 찾는 **SBR** 은 Sionna 의 **Mitsuba 3 엔진을 그대로** 쓰고, 그 위에 **PO 적분만 우리가** 얹습니다(SBR+PO).
> - **레이더 신호처리**(ECA/CFAR)는 🔴 우리가 짰습니다 — Sionna 에 레이더 DSP 가 없습니다.

### 3️⃣ 라이브러리 (실행 시점 **실측** 버전)

| 라이브러리 | 버전 | 무엇에 쓰나 |
|---|---|---|
| `sionna` | 2.0.1 | 광선추적(RT) + PHY(OFDM/NR/채널) — **이 프로젝트의 중심** |
| `mitsuba` | 3.8.0 | Sionna RT 의 렌더러·광선추적 백엔드 (OptiX, GPU). SBR 도 이걸 쓴다 |
| `drjit` | 1.3.1 | Mitsuba 의 JIT 컴파일러 — GPU 커널 생성 |
| `trimesh` | 4.12.2 | 메쉬 CAD·**검증** — 로프트/스윕/불리언 + watertight·법선·퇴화면 검사 |
| `scipy` | 1.18.0 | 스플라인(단면 보간·암 경로) · STFT(스펙트로그램) |
| `numpy` | 2.5.0 | 수치 계산 전반 |
| `matplotlib` | 3.11.0 | 도표 |

### 4️⃣ 어디서 돌렸나

- **Python** 3.12.13 · Linux 5.15.0-136-generic
- **GPU** — `src/gpu.py` 가 **여유 메모리를 보고 자동 선택**합니다 (하드코딩 없음):
  - 0, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 1, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 2, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 3, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
- `CUDA_VISIBLE_DEVICES` = (고정 안 함 — src/gpu.py 가 여유 메모리 보고 자동 선택)

- **계산 비용**: 5종 × 3밴드 RCS 는 GPU 한 장에서 수십 분(광선격자 λ/16). 마이크로도플러는 자세 144개 × SBR 재계산이라 드론당 수~십수 분.

### 5️⃣ 어떻게 다시 돌리나 (재현)

```bash
cd /home/yunjung/workspace/sionna2

# 5종 RCS(3밴드) + 재질 분해  -> report2_waveform_rcs.json
~/.venvs/py312/bin/python src/viz_report2.py

# 호버 rpm 유도 + 블레이드 마이크로도플러  -> report1.json
~/.venvs/py312/bin/python src/viz_report3.py

# JSON -> report08.ipynb (이 파일)
~/.venvs/py312/bin/python src/make_notebook08.py
```

### 6️⃣ 본문 숫자는 어디서 오나

이 노트북의 **숫자는 손으로 적지 않았습니다.** 측정 스크립트가 JSON 을 남기고, 노트북 생성기(`src/make_notebook*.py`)가 그 JSON 을 읽어 본문에 주입합니다. → **그림과 글이 어긋날 수 없습니다.** 숫자가 이상하면 JSON 을 보세요.

### 7️⃣ 무엇이 산출되나

| 산출물 | 무엇 |
|---|---|
| `outputs/report2_waveform_rcs.json` | **이 노트북의 RCS 숫자.** rcs(5종×3밴드) / materials(재질 분해) |
| `outputs/report1.json` | **이 노트북의 마이크로도플러 숫자.** articulation(호버 rpm) / microdoppler(지문) |
| `outputs/figures/report2_rcs_bars.png` | §2 5종 밝기 · 크기 추세 |
| `outputs/figures/report2_materials.png` | §3 재질 분해 (껍데기 vs 금속) |
| `outputs/figures/report2_rcs_polar.png` | §4 방위 패턴 (로브 vs 널) |
| `outputs/figures/report1_hover_rpm.png` | §5 호버 rpm 유도 |
| `outputs/figures/report1_microdoppler.png` | §5 블레이드 지문 + 가림 대가 |

### 8️⃣ ⚠️ 믿으면 안 되는 것 (신뢰 경계)

> 정직함이 이 프로젝트의 규칙입니다. **아래는 이 리포트가 보장하지 않는 것들입니다.**

- **절대 RCS 를 보장하지 않는다.** SBR 은 해석해(구·평판)로 검증되고(방법 검증은 report07), 드론 절대값은 §6 에서 **실측 문헌에 대조**했다 — 사과-대-사과 앵커(multiband Phantom 3 방위 선형평균)와는 **0.4~2.9 dB 안**에서 맞지만, 문헌 앵커 자체가 **12 dB 넘게 산포**해 절대 dBsm 은 **판정 보류**다. 이 리포트가 지지하는 것은 **상대 순서**(큰 기체가 밝다)와 **대역 추세**(밴드는 몇 dB만 움직인다)이지 특정 드론의 절대 dBsm 점값이 아니다.
- **플라스틱 셸의 밝기는 불확실 구간이다.** 1~3 mm 셸은 1.8~5.2 GHz 에서 **반투명**인데 first-hit SBR 은 셸을 뚫지 못한다. 그래서 진실은 '통드론'과 '셸 제거' 두 막대 사이에 있고, 그 간격 **0.3 dB** 는 측정오차가 아니라 **모델링 불확실도**로 읽어야 한다.
- **방위 패턴의 '널(골)'은 인용 금지.** 로브 사이 골은 격자밀도·대역평균·평활에 10 dB 넘게 흔들린다. **로브(봉우리)와 방위평균만** 믿는다.
- **호버 rpm 은 가정값이다.** 추력=무게 균형(C_T≈0.11)에서 유도한 물리 추정치이지 텔레메트리 실측이 아니다. flash·f_tip 은 이 rpm 에 선형으로 비례하므로, 실제 비행 rpm 이 다르면 지문 주파수도 그만큼 이동한다.
- **마이크로도플러는 슬로타임 모델이다.** 자세별 산란장은 SBR(Mitsuba 광선)로 재계산하지만, 블레이드 유연·와류 등 공기역학은 넣지 않았다. 지문의 **구조**(깜빡임·f_tip 경계)는 믿을 만 하나 절대 세기는 아니다.

### 9️⃣ 앞뒤 리포트

| 리포트 | 관계 |
|---|---|
| **앞** — [report07](report07.ipynb) | 이 숫자를 낸 **방법(SBR)** — 왜 옳은가·가림이 무엇인가 |
| **다음** — [report09](report09.ipynb) | 이제 탐지로. 먼저 챔버 **바닥이 놓는 함정**(표적 경유 유령) |

<details><summary><b>🔤 용어집 — 모르는 말이 나오면 여기</b> (클릭)</summary>

| 용어 | 뜻 |
|---|---|
| **RCS (σ)** | 레이더 되비침 밝기 [m²]. '이 표적이 얼마나 밝게 되쏘나'. dBsm = 10·log₁₀(σ/1 m²) |
| **dBsm** | 1 m² 대비 dB. −20 dBsm = 0.01 m² = 되비침이 사방 10 cm 판 만큼 |
| **광학영역** | 표적이 파장보다 훨씬 클 때. 밝기가 대략 **투영 넓이**를 따라가고 주파수엔 둔감 |
| **SBR / PO** | **SBR**=광선 쏴 보이는 면·가림 찾는 기하 단계(Sionna 의 Mitsuba 엔진 재사용). **PO**=그 밝은 면 위 산란장을 위상 맞춰 적분하는 물리 단계(=밝기). 스톡 Sionna 엔 PO 가 없어 우리가 얹음 |
| **가림(occlusion)** | 앞 부품에 막혀 안 보이는 면. 이걸 안 빼면 밝기·정지신호를 과대평가한다 |
| **로브 / 널** | 방위 패턴의 봉우리(로브)와 골(널). 로브는 안정, 널은 불안정 → 널은 인용 금지 |
| **마이크로도플러** | 표적의 **부분 운동**(프로펠러 회전)이 만드는 도플러 미세구조. 드론의 지문 |
| **flash rate** | 블레이드가 정면을 보여 번쩍이는 초당 횟수 = 날개수 × 회전수/60 |
| **f_tip** | 날개 끝 속도가 만드는 최대 도플러 폭. f_tip = 2·v_tip/λ·cos(el) |
| **pedestal(정지 몸통 신호)** | 회전 안 하는 몸통이 만드는 0 Hz 근처 강한 성분. 블레이드 깜빡임은 이 위로 솟아야 보인다 |
| **C_T (추력계수)** | 프로펠러 추력을 회전수로 잇는 무차원 계수. T = C_T ρ n² D⁴, 소형 로터 ≈0.11 |

</details>

---


## §1. 무엇을 왜 재는가 — 스톡 파이프라인의 공백

탐지는 표적이 레이더 눈에 **얼마나 밝은가(RCS, σ)**에서 출발한다. 그런데 스톡 Sionna 의 기본 광선추적(`PathSolver`)은 경로별 복소이득만 반환할 뿐 표적 표면 위 산란적분이 없어 σ 를 내지 못한다(→[report06](report06.ipynb)). ISAC 선행 연구는 이 밝기를 세 갈래로 다룬다 — 확산계수 S 가정, RCS 점표적 주입, 그리고 자작 SBR+PO. 우리는 세 번째, 선행이 실제로 쓰는 **SBR+PO** 방식으로 σ 를 계산했다(BVH SBR+PO, arXiv:2604.09243 과 같은 계열 — Sionna 가 쓰는 Mitsuba 3 광선엔진을 그대로 재사용하고 그 위에 PO 표면적분만 얹어 복소장 E 를 직접 낸다; 절차는 →[report07](report07.ipynb)).

이 리포트는 그 **결과**다 — 5종의 밝기(§2), 밝기가 어디서 나오나(§3), 방위 패턴(§4), 프로펠러 마이크로도플러 지문(§5). 다만 SBR+PO 의 해석해 검증(평판·구, report07)은 방법이 옳음만 보일 뿐 드론의 **절대 σ** 를 대조할 자체 기준이 파이프라인 안에 없다. 그래서 절대 스케일은 리포트 끝에서 **공개 실측 문헌 드론 RCS 로 앵커**한다(§6).

## §2. 5종 얼마나 밝나 — 밝기를 정하는 건 크기지 주파수가 아니다

![rcs bars](outputs/figures/report2_rcs_bars.png)

각 드론을 **360° 다 돌려가며**(방위) 세 통신대역에서 재고, 그 **방위평균**을 밝기 대표값으로 씁니다. (봉우리 값이 아니라 평균입니다 — 링크버짓에 넣을 정직한 숫자는 이쪽입니다.)

**밴드평균 방위평균 RCS [dBsm]** (el = 15°, 격자 λ/16):

| 드론 | 대각 [mm] | 무게 [g] | LTE 1.8 GHz | 5G NR 3.5 GHz | WiFi 5.2 GHz |
|---|---|---|---|---|---|
| DJI Mini 5 Pro | 275 | 250 | -25.5 | -22.3 | -21.6 |
| DJI Mavic 4 Pro | 441 | 1063 | -16.8 | -18.4 | -16.0 |
| DJI Matrice 4E | 438.8 | 1219 | -21.6 | -22.2 | -20.2 |
| DJI S1000+ | 1045 | 9500 | -17.9 | -13.9 | -12.9 |
| DJI Phantom 4 | 350 | 1380 | -21.7 | -18.9 | -18.9 |

**두 가지가 한눈에 보입니다.**

1. **크기가 밝기를 정합니다.** 가장 큰 DJI S1000+(대각 1045 mm, 9.5 kg 8로터)가 가장 밝고, 가장 작은 DJI Mini 5 Pro(275 mm, 250 g)가 가장 어둡습니다. 둘 사이가 **8.2 dB** — 퍼센트 수준이 아니라 **약 6.6 배**(선형 σ 비) 차이입니다. 오른쪽 산점도가 대각↔밝기 추세를 그대로 보여줍니다.

2. **대역(주파수)은 별로 안 움직입니다.** 같은 드론을 1.8 → 5.2 GHz 로 옮겨도 밝기는 평균 **3.2 dB** 밖에 안 변합니다. 드론이 이미 파장(λ = 6~17 cm)보다 훨씬 커서 **측정한 좁은 대역(1.8~5.2 GHz) 안에서는** 밴드 스윙이 작기 때문입니다 — 이 범위에선 밝기가 대략 **투영 넓이**를 따라가고 파장엔 둔감합니다.

> **크기 순서가 무게 순서와 살짝 다른 이유.** RCS 는 무게가 아니라 **되비추는 금속 표면**이 정합니다. DJI Phantom 4는 무겁지만(1.38 kg) 몸체가 매끈해 측면 로브가 좁고, DJI Mini 5 Pro는 250 g 급이라 되쏠 금속 자체가 작습니다. 순서는 **투영된 금속 넓이** 쪽을 따릅니다.

> **링크버짓으로 가져갈 숫자는 방위평균입니다.** 봉우리(peak)는 순간적으로 더 밝지만 방위가 조금만 틀어져도 사라집니다 — 탐지 성능을 보수적으로 보려면 평균을 씁니다.

## §3. 밝기는 속 금속이 지배한다 — 껍데기는 스크린이다

![materials](outputs/figures/report2_materials.png)

밝기가 **어디서** 나오는지 보려면 드론을 부품별로 벗겨가며 재보면 됩니다. DJI Mavic 4 Pro 한 대를 3.5 GHz 에서, 부품을 하나씩 지우며 방위평균 밝기를 다시 쟀습니다(밝은 쪽이 위):

| 무엇을 남겼나 | 방위평균 RCS | 통드론 대비 |
|---|---|---|
| **통드론** (플라스틱 셸 포함) | -18.41 dBsm | 기준 |
| 셸 **제거** (전파가 플라스틱을 통과) | -18.11 dBsm | **+0.30 dB** |
| 프로펠러만 제거 | -18.37 dBsm | +0.05 dB |
| **금속 코어만** (모터+배터리+PCB+카메라) | -18.02 dBsm | **+0.39 dB** |
| 유전체만 (금속 하나도 없이) | -25.27 dBsm | -6.86 dB |

**세 줄로 요약됩니다.**

- **플라스틱 껍데기를 지웠더니 오히려 +0.30 dB 밝아졌습니다.** 셸은 밝기에 거의 기여하지 않으면서, 뒤에 있는 금속으로 갈 광선을 약하게 가로막던 **가림막**이었기 때문입니다. (지운다는 건 페인트를 칠하는 게 아니라 그 면을 메쉬에서 **삭제**해 전파가 통과하게 하는 것입니다.)
- **금속 코어만 남겨도 +0.39 dB** — 통드론과 사실상 같습니다. 반대로 **금속을 전부 빼면 -6.86 dB** 어두워집니다. **밝기를 만드는 건 속 금속이고, 플라스틱은 조연**입니다.
- 프로펠러(플라스틱)는 정지 상태에서 **+0.05 dB** — 밝기엔 거의 무의미합니다. (단, **돌면** 이야기가 완전히 달라집니다 → §5.)

> ⚠️ **정직한 한계 하나.** 1~3 mm 플라스틱 셸은 1.8~5.2 GHz 에서 실제로는 **반투명**입니다 — 전파가 얼마쯤 통과합니다. 그런데 우리 SBR 은 광선이 **첫 충돌에서 멈추므로** 셸을 뚫지 못합니다. 그래서 진실은 '통드론(불투명 셸)'과 '셸 제거(투명 셸)' **두 막대 사이 어딘가**에 있습니다. 그 간격 **0.3 dB** 는 측정오차가 아니라 **모델링 불확실도**로 읽으십시오.

> 두 엔진(전파용 Sionna RT · RCS용 SBR)이 **같은 재질표**(`src/materials.py`)를 읽습니다. 오른쪽 표의 반사계수가 그것 — 조용히 어긋날 수 없습니다.

## §4. 방위 패턴 — '봉우리'는 인용, '골'은 인용 금지

![rcs polar](outputs/figures/report2_rcs_polar.png)

드론을 한 바퀴 돌리면 밝기는 방위에 따라 **꽃잎 모양**으로 오르내립니다. 넓은 금속면이 정면으로 보이는 방위에서 **봉우리(로브)** 가 서고, 그 사이에서 **골(널)** 로 떨어집니다.

**봉우리가 어디에 서는지는 기체 대칭성이 정합니다 — 5종 공통이 아닙니다.** 코(0°)·꼬리(180°)·측면(90°/270°) 네 방향에서 고르게 서는 것은 직사각 대칭 쿼드(DJI Phantom 4)뿐입니다: 네 방위 ±15° 구간 평균이 방위평균보다 각각 +6.4 / +6.3 / +6.4 / +6.3 dB 로, 서로 0.11 dB 안에 모입니다. 반면 DJI Mini 5 Pro는 코·꼬리가 방위평균보다 오히려 −5.3 / −5.5 dB **낮고** 측면(+5.3 / +5.4 dB)에서만 섭니다. 접이식 암(Mavic 계열)·8암 옥토는 네 방향 대칭을 보이지 않습니다 — 3.5 GHz 최강 로브 방위도 기체마다 다릅니다(DJI Mini 5 Pro 77°, DJI Mavic 4 Pro 180°, DJI Matrice 4E 353°, DJI S1000+ 93°, DJI Phantom 4 271°).

**여기서 반드시 지켜야 할 규칙:**

- **봉우리(로브)는 대역평균(5개 주파수)과 3° 평활을 거친 뒤에는 인용해도 됩니다.** 평활 전 단일 주파수의 개별각은 격자를 반절해도 평균 ~2 dB 흔들립니다(`report_mesh/outputs/mesh_verify.json` H·I). 링크버짓의 '최선의 경우'로 쓸 수 있습니다.
- **골(널)은 절대 인용하지 마십시오.** 골의 깊이는 여러 반사가 서로 상쇄돼 생기는 것이라, 격자를 조금만 바꿔도 **10 dB 넘게** 출렁입니다. '이 각도에서 −40 dBsm 으로 안 보인다' 같은 주장은 하면 안 됩니다.

> 그래서 이 리포트가 밖으로 내보내는 숫자는 **§2 의 방위평균**과 **로브 높이**뿐입니다. 특정 방위의 널 깊이는 내부 그림에서만 봅니다.

> 각 곡선은 361개 방위 × 대역 내 5개 주파수 평균 × 3° 평활입니다(el = 15°). 큰 기체(S1000+)일수록 로브가 잘게 갈라지는 건 전기적 크기(size/λ)가 커서 로브가 촘촘해지기 때문입니다.

![.](outputs/renders/anim/rcs_azimuth_matrice4e.gif)

<sub>Matrice 4E RCS 방위각 폴라 — 각도마다 수 dB~수십 dB 출렁인다(SBR 결과).</sub>

## §5. 프로펠러 지문 — 마이크로도플러

지금까지는 드론이 **가만히** 있을 때의 밝기였습니다. 하지만 드론의 프로펠러는 초당 수십 바퀴를 돕니다. 돌아가는 블레이드는 **정면을 보일 때마다 반사가 번쩍**이고, 날개 끝은 시속 200 km 급으로 움직여 큰 도플러(주파수 변화)를 만듭니다. 이 미세구조가 드론을 새·잡음과 가르는 **지문**입니다.

다중 프로펠러 드론의 **바이스태틱 마이크로도플러**를 모델링하는 것은 선행 연구가 이미 측정으로 검증해 둔 접근입니다 — Costa & Thomä(TU Ilmenau, IEEE J-STEAP 2025, arXiv:2504.05168)는 프로펠러를 thin-wire 점산란체 + PO 로터 RCS 로 놓고 분산 ISAC OFDM 에서 실측 대조했습니다. 우리는 같은 목표를 **전체 메쉬 SBR** 로 풀어(점산란체 근사 없이 가림까지 포함) 아래 지문을 얻습니다.

### 5.1 지문의 두 눈금은 호버 회전수에서 나온다

![hover rpm](outputs/figures/report1_hover_rpm.png)

지문에는 두 개의 눈금이 있습니다.

- **flash rate(번쩍임 주기)** = 날개수 × 회전수/60. 2엽 프로펠러는 한 바퀴에 정면을 **두 번** 보이므로 flash = 회전수/30. → 프로펠러가 **크고 느린** 기체는 드물게, **작고 빠른** 기체는 자주 번쩍입니다.
- **f_tip(날개끝 도플러 폭)** = 2·v_tip/λ·cos(el). 날개 끝 속도 v_tip = ω·R 가 만드는 **최대** 도플러입니다. 모델 안에서 이보다 빨리 움직이는 산란체는 없으므로, 진짜 마이크로도플러는 **±f_tip 안에** 갇힙니다.

둘 다 **호버 회전수**만 알면 정해집니다. 회전수는 텔레메트리가 없으니 **물리로 유도**합니다 — 호버란 4(또는 8)개 로터의 추력이 정확히 무게를 받치는 상태이고, 추력은 T = C_T ρ n² D⁴ (C_T ≈ 0.10~0.12) 로 회전수 n 과 이어집니다. 무게와 프로펠러 지름 D 를 넣어 n 을 풀면 **아래 표의 호버 rpm**이 나옵니다(가정값이지만 물리 범위 안입니다).

**5종의 프로펠러 지문** (3.5 GHz, el = 15°):

| 드론 | 로터 수 | 호버 rpm | flash [Hz] | f_tip [kHz] | 가림 이득 [dB] |
|---|---|---|---|---|---|
| DJI Mini 5 Pro | 4 | 5500 | 183 | ±0.99 | 17 |
| DJI Mavic 4 Pro | 4 | 3600 | 120 | ±1.14 | 39 |
| DJI Matrice 4E | 4 | 3800 | 127 | ±1.23 | 10 |
| DJI S1000+ | 8 | 3600 | 120 | ±1.62 | 10 |
| DJI Phantom 4 | 4 | 5500 | 183 | ±1.56 | 16 |

→ **flash rate 는 프로펠러 크기·회전수를 그대로 반영합니다.** 큰 프로펠러(S1000+ 15인치)는 느리게 돌아 120 Hz, 작은 프로펠러(Mini 5 Pro 6인치)는 빠르게 돌아 183 Hz 로 번쩍입니다.

> ⚠ **그러나 flash 하나로는 기체가 갈리지 않습니다.** 위 표의 5종이 내는 flash 는 **3개 값뿐**이고 **DJI Mini 5 Pro = DJI Phantom 4** · **DJI Mavic 4 Pro = DJI S1000+** 가 각각 같은 값에서 겹칩니다. flash = 날개수 × rpm/60 이라 로터 수·프로펠러 지름이 달라도 호버 rpm 이 같으면 같은 값이 나오기 때문입니다. 겹친 짝은 f_tip 이 갈라 줍니다(예: DJI Mavic 4 Pro ±1.14 kHz vs DJI S1000+ ±1.62 kHz) — 즉 지문은 **(flash, f_tip) 한 쌍**이지 flash 단독이 아니고, 두 눈금 모두 호버 rpm 가정에 비례합니다.

### 5.2 지문을 보려면 '가림'이 필수다

![microdoppler](outputs/figures/report1_microdoppler.png)

위 그림의 각 판은 시간(가로) × 도플러(세로)로 그린 **슬로타임 반사장**입니다. 프레임마다 블레이드 자세를 다시 놓고 SBR 로 산란장을 새로 계산합니다 — 세로 줄무늬가 바로 블레이드 번쩍임, 파란 점선이 ±f_tip 경계입니다.

여기서 **가림이 왜 필수인지**가 오른쪽 아래 막대에 있습니다. 블레이드가 몸통 뒤로 돌아가면 **안 보여야** 하는데, 가림을 안 하는 순수 PO 는 **몸통에 가려 안 보이는 날개까지 다 세어** 0 Hz 근처의 **정지 몸통 신호(pedestal)를 부풀립니다.** 그 부풀림이 드론마다 **10~39 dB** — 그만큼 블레이드 깜빡임이 몸통 신호 아래 묻힙니다.

SBR 은 광선이 **첫 충돌에서 멈춰** 가림이 공짜라, 부풀린 pedestal 을 걷어내고 **깜빡임을 몸통 위로 되살립니다.** 그래서 §3 에서 '정지 상태 프로펠러는 밝기에 무의미'했지만, **돌면** 프로펠러가 지문의 주역이 됩니다 — 정지 밝기가 아니라 **시간에 따른 변조**가 정보이기 때문입니다.

> ⚠ **범위 주의.** 이 pedestal 비교는 **가림(몸통 뒤 숨은 블레이드)만** 격리하려 first-hit SBR(투과 미적용)을 쓴다. **셸 속 내부 금속**(배터리·PCB)은 실재하는 정적 산란체라 pedestal 에 정당히 들어가야 하며, 헤드라인 σ 엔진은 그걸 **셸 투과로 되살린다**(§3) — 즉 '내부 금속을 걷어내는 것'은 가림의 역할이 아니다. (sbr_field·다중반사 경로의 투과 일관화는 후속 과제.)

> ⚠ **선행 실측과의 방향 — 아직 확정 아님(정직하게).** 이 분야 **유일한 바이스태틱 마이크로도플러 실측**(Costa·Thomä, TU Ilmenau, RadarConf24)은 스펙트럼에서 **DC(정지 몸통)가 블레이드 선보다 ≈47 dB 우세**하다고 판독된다 — 즉 정지 몸통이 매우 지배적이다. 우리 JSON 의 DC↔AC 비(|DC|/std(AC))는 **SBR -2.2~+16.4 dB** 인 반면 가림을 안 한 **PO 는 +17.3~+37.2 dB** 로, **PO 쪽이 그 실측(강한 몸통 우세)에 더 가깝다.** 즉 SBR 가림이 pedestal 을 낮춰 깜빡임을 살리는 방향은 **이 한 실측에서는 멀어지는 방향**이다. 다만 (ⓐ 정의가 다르고 — 우리 값은 시간영역, 실측은 스펙트럼 선 피크, 선-피크 재계산은 다음 단계, ⓑ 실측은 4개 중 1개 로터만 회전·탄소 골격·β=60°·편파 미기재) **방향은 보이나 확정은 아니다.** 가림 자체의 기하 타당성(몸통 뒤 블레이드는 안 보인다)은 그대로 유지하되, 'PO 과대·SBR 교정' 이라는 **절대 세기 판정은 유보**한다.

> **직관 하나.** 선풍기 날개에 손전등을 비추면, 날개가 정면을 보이는 순간마다 규칙적으로 반짝입니다. 그런데 날개가 **선풍기 몸통 뒤로** 넘어가는 동안은 안 보이죠(가림). 이 '보였다 안 보였다'가 규칙적 반짝임을 만듭니다. 몸통 뒤 날개까지 억지로 세면(가림 무시) 밋밋한 몸통 밝기만 커져서 정작 반짝임이 안 보입니다.

> ⚠️ 호버 rpm 은 추력 균형에서 유도한 **가정값**입니다. flash·f_tip 은 rpm 에 비례하므로, 실제 비행 회전수가 다르면 지문 주파수도 그만큼 이동합니다. 지문의 **구조**(깜빡임·±f_tip 경계·기체별 순서)는 믿을 만하나 절대 주파수는 rpm 가정에 달려 있습니다.

In [ ]:
# §2 재현 — 한 드론의 밝기를 한 대역에서 직접 재본다 (SBR)
import numpy as np
from rcs_po import drone_rcs_pattern_bw, dbsm      # 기본 엔진은 'sbr'

az = np.arange(0, 361, 2.0)
sig, n_rays = drone_rcs_pattern_bw('s1000plus', 5.21e9, 80e6, az, el_deg=15.0, n_f=5)
print(f'방위당 광선 {n_rays:,}발  (격자 lambda/16)')
print(f'S1000+ @ 5.2 GHz  방위평균 {dbsm(np.mean(sig)):+.2f} dBsm  '
      f'(로브 최대 {dbsm(np.max(sig)):+.2f} dBsm)')

In [ ]:
# §5 재현 — 호버 rpm 유도 + 블레이드 지문의 두 눈금
#   flash = blades * rpm/60,   f_tip = 2*v_tip/lambda * cos(el)
import numpy as np

specs = dict(mini5pro=(0.2499,0.1524,4), mavic4pro=(1.063,0.267,4),
             matrice4e=(1.219,0.274,4), s1000plus=(9.5,0.381,8),
             phantom4=(1.38,0.240,4))       # (질량 kg, 프로펠러 지름 m, 로터 수)
rho, CT, blades = 1.225, 0.11, 2
lam, el = 3e8/3.5e9, np.deg2rad(15.0)
for d,(m,D,nr) in specs.items():
    T = m*9.81/nr                                  # 로터당 추력 = 무게/로터수
    n = np.sqrt(T/(CT*rho*D**4))                   # T = CT rho n^2 D^4  ->  n [rev/s]
    rpm = n*60
    flash = blades*rpm/60
    v_tip = (2*np.pi*n)*(D/2)
    f_tip = 2*v_tip/lam*np.cos(el)
    print(f'{d:10s} rpm~{rpm:5.0f}  flash {flash:5.1f} Hz  f_tip +-{f_tip/1e3:4.2f} kHz')
# ↑ 같은 물리(T=CT·rho·n^2·D^4)에서 나온다. 단 report1.json 은 로터별로 CT 를 세밀 보정하므로,
#   이 고정 CT=0.11 스니펫은 자릿수 수준의 예시일 뿐 §5.1 표값과 정확히 일치하지는 않는다.

---
## §6. 선행 연구의 방식과 실측 대조 — 절대값 검증

해석해(평판·구, report07)는 SBR+PO 라는 **방법**이 옳음을 보이지만 드론의 **절대 σ** 는 보장하지 못한다. 절대 스케일의 기준은 선행 연구가 남긴 **실측 문헌 드론 RCS** 다.

ISAC 문헌에서 표적 밝기는 세 갈래로 처리된다 — **(b) 확산계수 S 가정**(Great-X arXiv:2507.08716 · Deterministic-Modeling arXiv:2603.28736, EuCAP 2026), **(c) RCS 상수 주입**(3GPP · 오픈 MATLAB arXiv:2606.07328), **(d) 자작 SBR+PO / 산란 add-on**(Sionna-RT 확장 계열). 우리는 **(d)** 를 택했다 — 소형 드론은 확산 S 실측 보정 데이터가 없고 부위별 재질 차이가 RCS 를 지배하기 때문이다.

⚠ **이 계열의 대표 선행은 우리 방법을 명시적으로 비판한다.** Ziganshin(arXiv:2604.05991)은 서론에서 SBR+PO 를 자기 방법의 **대척점**으로 놓는다 — *"This SBR+PO approach, however, is limited to the illuminated region and is not suitable to predict the scattered field in the shadow region of the obstacle. Furthermore, the need to cascade PO after RT negates the computational advantages of RT."* 그들은 Sionna-RT 솔버 자체를 UTD+정점회절로 확장해 **PEC 차량·구(2–10 GHz, facet E>1.5λ)** 를 다루고, 우리는 그 솔버가 쓰는 Mitsuba 광선엔진 위에 PO 를 얹어 **few-λ 부위별 유전체 소형 드론**을 다룬다 — UTD 유효조건(E>1.5λ)이 성립하지 않는 영역이라 방법 선택이 갈린다. 그리고 그들이 지적한 **그늘영역·상반성 한계는 우리도 이미 자발적으로 공개한다**: report07 §5 는 이 리포트의 σ 가 *전방산란(β→180°)에서 σ≡0, 깊은 널에서 상반성 σ(û_i,û_s)=σ(û_s,û_i) 붕괴* 하는 **모노스태틱 등가값**임을 명시한다(물리적 인식은 있고, 필요한 것은 인용 프레이밍이다). 상용 CADFEKO(LAMBDA arXiv:2607.03826)·비공개 RadarSimPy·독립엔진 BVH SBR+PO(arXiv:2604.09243) 대신 **선행이 실제로 쓰는 자작 SBR+PO** 를 따랐고, 검증은 라이브러리 대조가 아니라 아래 **실측 문헌 앵커**로 세운다(근거: `prior_work/pw01`).

우리가 쓰는 신형(Mavic 4 Pro·Matrice 4E)의 실측 RCS 는 아직 논문에 없습니다(2024~25 출시). 절대 판정의 **1급 근거는 우리 세 밴드를 전부 커버하는 Phantom 3 실측**이다(아래 첫 표). 밴드나 지표가 어긋나는 나머지 실측(Li & Ling·Ezuma·Semkin·Quevedo)은 **방향성 참고**로만 쓴다.

#### 1급 절대앵커 — DJI Phantom 3, 우리 세 밴드를 전부 커버

우리 phantom4 는 Phantom 3 와 **대각이 정확히 같은 350 mm DJI 쿼드**다. Phantom 3 는 두 편이 **같은 측정 캠페인**(Wei Fan/Southeast Univ. 데이터)을 서로 다른 평균 규약으로 요약해, 우리 세 밴드(1.8/3.5/5.2 GHz)를 모두 덮는 유일한 실측 앵커다:

| 앵커 (Phantom 3, 350 mm) | 평균 규약 | @1.8 | @3.5 | @5.2 GHz | 출처 |
|---|---|---|---|---|---|
| multiband (Das 2026) | **선형** ★우리와 동일 | −18.80 | −18.46 | −18.10 | multiband · IEEE WCL 15:3731 · Table III · [PDF] |
| mono3d (Yuan 2025, 같은 캠페인) | dB영역 | −15.57 | −15.05 | −14.51 | mono3d · EuCAP 2025 · IV절 · [PDF] |
| **우리 phantom4** (방위 선형평균) | 선형 | −21.69 | −18.85 | −18.90 | JSON |
| **Δ (우리 − multiband, 선형↔선형)** | 사과-대-사과 | **−2.89** | **−0.40** | **−0.81** | Δ [dB] |
| Δ (우리 − mono3d, 선형↔dB영역) | 규약 미정렬 | −6.12 | −3.81 | −4.39 | Δ [dB] |

<sub>**판정 (P0 — 평균 규약 열).** 평균 규약까지 맞춘 **multiband 행이 유일한 사과-대-사과**다 — 우리 `mean_dbsm` 은 방위 **선형평균**(10log₁₀(mean σ), `viz_report2.py:841`)이고 multiband §III-1 도 선형평균이다. 그 앵커와 우리 350 mm 기체는 3.5/5.2 GHz 에서 0.4/0.8 dB, 1.8 GHz 에서 2.9 dB 안에서 맞는다(우리가 약간 어두운 쪽). **mono3d 는 같은 측정**을 dB영역 평균으로 요약해 3.4 dB 위로 나오는데, 이는 로그정규에서 (선형평균 − dB영역평균) = (ln10/20)·ε² 로 설명되는 **순수 규약 차**다(ε≈5.2 dB → ≈3.2 dB, 노트 §2-2, PDF 확인). 즉 이 3.4 dB 가 **절대판정의 하한 불확도**이며, 규약을 안 밝힌 두 요약을 그냥 병치하면 우리가 3.4 dB 만큼 자의로 밝거나 어두워 보인다. ⚠ 아직 정렬 안 된 축: 앙각(우리 el=15° ↔ 문헌 el=0° 수평면)·편파(스칼라 Γ ↔ co-pol). **el=0° 문헌 대조컷은 아래 **§6.1(선행 방법론 정량 대조)** 에서 5기종×3밴드로 산출했다.**</sub>

이 1급 앵커와 견주면 우리 절대 레벨은 **수 dB 안**이다. 그런데 아래 방향성 참고표의 Li & Ling **aspect-peak**(−27.5 dBsm)는 위 두 실험실의 **방위 mean**(−18.5 / −15.0)보다도 9~12 dB **아래**다 — peak 가 다른 실험실 mean 보다 낮을 수는 없다(자세-peak ≥ 방위-mean). 즉 **문헌 절대앵커끼리가 이미 물리적으로 불가능한 방향으로 12 dB 넘게 어긋나 있고**, 이 산포가 우리 오차보다 크다. 그래서 **절대 dBsm 판정은 보류**하고, 아래 표는 방향성 참고로만 읽는다:

| 문헌 (실측) | 밴드 | 측정 RCS | 우리와의 관계 |
|---|---|---|---|
| **Li & Ling 2017** (IEEE AWPL, ~99인용) · 등급 **[N]**(PDF 부재) | **3–6 GHz** ★밴드일치 | Phantom 2(350 mm) **−27.5**, 3DR Solo(460) −24.2, Inspire 1(560) −13.7 dBsm (모두 **aspect-peak**, 자세 스프레드 ~14 dB) | 지표는 우리 peak 와 맞지만(peak↔peak, 대각 짝) **절대 판정엔 못 쓴다** — 이 peak(−27.5)가 위 1급 앵커의 mean 보다 12 dB 아래라 (peak<mean, 물리 불가) 절대교정이 낮다. peak↔peak 로 재면 우리가 **+8.1~+15.8 dB**(대각비 ±10% 짝, 최정합 350 mm 짝 +15.8 dB)로 나오지만 이는 **Li & Ling 의 낮은 교정 탓**이지 우리가 밝다는 증거가 아니다(같은 기체 mean 은 1급 앵커와 −0.4 dB 로 맞는다). 짝짓기 원장은 아래 |
| Ezuma 2019 (compact-range) · 등급 **[N]** | 15 / 25 GHz | Phantom 4 Pro −15.0 / −12.4 dBsm | 밴드갭이 커서 **절대값 대조는 하지 않는다**. **기울기 대조도 하지 않는다** — 우리 1.8~5.2 GHz 는 3점뿐이라 회귀가 구속되지 않는다(R² 0.13~0.88, mavic4pro 0.13). 인용하는 것은 §2 의 **밴드 스윙**(3.2 dB)뿐 |
| Semkin 2020 (IEEE Access) · 등급 **[N]** | 26–40 GHz | Mavic Pro(335 mm 플라스틱) −16.8, Phantom 4 Pro −16.4, Matrice 100(650 mm 카본) −10.5 dBsm | **규약만 차용**(재질·로터 정지·편파 HH 를 명기하는 보고 방식). **재질 이득 수치는 인용하지 않는다** — M100 은 카본인 동시에 거의 2배 크고, 광학영역 면적 스케일링만으로 20·log₁₀(650/335) = **+5.8 dB** 라 두 기체 차이의 대부분이 크기 효과다 |
| Quevedo 2019 (IET RSN) · 등급 **[N]** | X-band 8.75 GHz | Phantom 4 −20~−4.6 dBsm(프롭 회전 의존) | 범위가 15 dB 폭이라 **정량 대조에는 쓰지 않는다**. 차용하는 것은 방향성뿐 — 프로펠러 회전이 σ 를 크게 흔든다(우리 마이크로도플러 서사) |

**짝짓기 원장** (Li & Ling peak↔peak — 방향성 참고) — 위 peak↔peak 범위가 어느 짝에서 나왔는지, 그리고 크기 불일치를 이 절이 Semkin 행에서 쓰는 것과 **같은 면적 스케일링**으로 뺐을 때 무엇이 남는지 그대로 편다. ⚠ 이 표는 **Li & Ling 절대교정이 위 1급 앵커보다 낮다**는 전제 위에 있어 절대 판정이 아니라 **크기-순서 재현 확인용**이다:

| 우리 기체 | 대각 (mm) | 짝 (Li & Ling) | 대각 (mm) | 대각비 | Δpeak | 크기보정 20log₁₀(비) | 보정 후 |
|---|---|---|---|---|---|---|---|
| DJI Mini 5 Pro ⚠ | 275 | DJI Phantom 2 | 350 | ×0.79 | **+11.54 dB** | −2.09 dB | +13.63 dB |
| DJI Mavic 4 Pro | 441 | 3DR Solo | 460 | ×0.96 | **+11.14 dB** | −0.37 dB | +11.51 dB |
| DJI Matrice 4E | 439 | 3DR Solo | 460 | ×0.95 | **+8.07 dB** | −0.41 dB | +8.48 dB |
| DJI S1000+ ⚠ | 1045 | DJI Inspire 1 | 560 | ×1.87 | **+8.01 dB** | +5.42 dB | +2.59 dB |
| DJI Phantom 4 | 350 | DJI Phantom 2 | 350 | ×1.00 | **+15.79 dB** | +0.00 dB | +15.79 dB |

<sub>⚠ 표시한 2행은 **짝이 아니다** — 문헌의 최대 기체가 560 mm 라 그보다 크거나 훨씬 작은 우리 기체에는 대응 실측이 없다(대각비 DJI Mini 5 Pro ×0.79·DJI S1000+ ×1.87). 그래서 peak↔peak 범위 +8.1~+15.8 dB 는 대각비 ±10% 안인 3행에서만 잡았다. 이 표가 말하는 것은 <b>크기 순서가 재현된다</b>는 것뿐이다 — 벗어난 행까지 면적 스케일링으로 보정해도 부호가 유지된다(최솟값 +2.6 dB). (광학영역 σ∝면적 가정을 few-λ 영역에 그대로 적용한 거친 보정이라 <b>보정값 자체는 인용하지 않고</b> 부호 확인용으로만 쓴다.)</sub>

<sub>**정리 — 절대 σ 는 판정 보류, 상대 결론만.** (1) 밴드·지표·기하·**평균규약**을 모두 맞춘 유일한 사과-대-사과는 위 **1급 앵커(multiband Phantom 3, 방위 선형평균)**다. 그와 우리 350 mm 정합기는 3.5/5.2 GHz 에서 0.4/0.8 dB, 1.8 GHz 에서 2.9 dB **안**에서 맞는다. Li & Ling peak↔peak 로는 +8.1~+15.8 dB '위'로 나오지만 그 앵커의 절대교정이 1급 앵커 mean 보다 12 dB 낮다(peak<mean, 물리 불가)는 것이 확인되므로 **절대 밝기 증거가 아니다**. (2) **낮은 밴드에서 우리가 오히려 어둡다는 신호도 있다** — 우리 1.8 GHz 는 1급 앵커보다 2.9 dB 어둡고, 같은 측정을 두 논문이 요약한 값이 3.4 dB(순수 평균규약 차) 벌어진다. Li & Ling 12–15 GHz 하강분으로 외삽하면 3.5 GHz 진값이 −25~−28 dBsm 쪽이어야 한다는 논리도 있으나, 이는 **1급 앵커의 직접 실측(3.5 GHz −18.46) 과 정면충돌**한다(우리 mavic4pro −18.36 dBsm 과 거의 같다) — 즉 그 외삽은 신뢰할 수 없고, 절대오차의 **방향은 단정할 수 없다**. 이 모든 어긋남이 **앵커 산포 > 우리 오차** 라는 한 사실을 가리킨다. (3) ⚠ **아직 정렬되지 않은 축.** 앙각 — 우리는 el = 15°, **문헌은 전부 수평면(el=0°)**(mono3d θ={90,0,180}°·unified-rcs 'elevation fixed at 90°'·multiband 수평 원호) → **미정렬**이며, el=0° 문헌 대조컷은 아래 **§6.1(선행 방법론 정량 대조)** 에서 5기종×3밴드로 산출했다. 편파 — 우리 SBR 은 스칼라 Γ 라 co-pol/cross-pol 을 분리하지 않고(`src/rcs_sbr.py`) 문헌은 co-pol 측정이다. 지표 — 우리 peak 는 방위 361점·대역 내 5주파수 평균의 **평활 전** 최대다. 출처 등급 — Li & Ling·Ezuma·Semkin·Quevedo 는 전부 **[N]**(워크스페이스 노트 근거, PDF 부재)이라 절대 판정에 못 쓰고, 1급 앵커 multiband·mono3d 만 **PDF 확인**이다. (4) 그래서 절대 dBsm 은 판정을 **보류**하고, 검출은 **σ 밴드**로 제시해 상대 결론(모드·파형 비교)이 밴드 전체에서 흔들리지 않음을 보인다. 서지 노트: `refs/drone_papers/` · 1급 앵커 PDF: `paper_sionna_Ray/`.</sub>

### 절대값 앵커 — 실측 문헌 RCS 와 교차검증

밴드가 더 가까운 실측(2.4~4.5 GHz)과도 자릿수를 맞춰 본다. 값 출처는 `prior_work` 파일럿 조사이고, **등급을 행마다 표기**한다 — [N]은 워크스페이스 노트 근거, [W]는 웹 메타데이터 근거(원문 미확인)다:

| 실측 (동종 드론) | 밴드 | 측정 RCS | 출처 · 등급 |
|---|---|---|---|
| DJI Mavic Pro | **2.4 GHz** ★밴드·기체급 근접 | **≈ −15.2 dBsm** (0.03 m²) | Güvenç/NCSU 서베이(arXiv:2402.05909) · [W] |
| DJI Mavic Pro | 15 / 25 GHz | −17.1 / −16.2 dBsm | Ezuma/Güvenç(arXiv:1911.05926) · [N] |
| DJI Phantom 4 Pro | 15 / 25 GHz | −15.0 / −12.4 dBsm | 〃 |
| 소형기(바이스태틱, 무향실) | 2.75 / 4.51 GHz | −9.8→−5.3 / −7.8→−5.0 dBsm | Frankford/Björklund(IET RSN) · **[W] 원문 미확보(paywall)** |

<sub>**교차검증 판정 — 행마다 자격이 다르다.** (1) **Mavic Pro @2.4 GHz** 행은 소형 쿼드·sub-6 이라 크기·밴드 어느 쪽으로도 실격 사유가 없다. 우리 mavic4pro 3.5 GHz 는 방위평균 −18.36 / 봉우리 −13.06 dBsm 이므로 문헌값 −15.2 dBsm 은 우리 방위평균보다 +3.2 dB, 우리 봉우리보다 −2.1 dB 다. ⚠ 단 그 서베이 값의 **지표 정의(peak/mean)가 미상**이고 기체 세대가 달라, 부등호를 세우지 않고 **자릿수 sanity check** 로만 쓴다. (2) **소형기 바이스태틱** 행은 원문이 paywall 이라 기종·자세·편파·지표를 확인하지 못했다 — 방향성 참고로만 쓴다. (3) **15 / 25 GHz** 두 행은 밴드갭이 커서 절대값 대조에 쓰지 않는다. ⚠ 공통 한계: 우리 SBR 은 스칼라 Γ 라 **편파를 분리하지 않고**(`src/rcs_sbr.py`) 문헌은 co-pol 측정이며, 앙각·자세 규약도 다르다. 즉 이 표는 자릿수 확인이지 점일치 검증이 아니고, **절대 레벨의 사과-대-사과 근거는 밴드·지표·기하·평균규약이 모두 정렬된 위 1급 앵커(multiband Phantom 3)뿐**이다. 그 대조가 말하는 것은 **크기 순서·자릿수·대역 추세는 재현되고 절대 레벨은 그 앵커와 수 dB 안에서 맞지만, 문헌 앵커 자체의 산포(12 dB↑)가 우리 오차보다 커 절대 dBsm 판정은 보류**한다는 것이다.</sub>

---
### §6.1. 선행 방법론을 그대로 차용한 정량 대조 (원시 σ 로)

위 §6 은 **평활·대역평균 후** 값(`sigma_smooth`)으로 절대 레벨을 견줬다. 여기서는 정독노트 §4-P1(9~16)이 지목한 선행 절차 — **분포적합·μ/ε 회귀·금속구 교정·el컷·분위점** — 을 그대로 가져와 우리 σ 를 문헌과 **정량비교**한다. ⚠ **입력은 전부 원시 σ(az)**다 — 단일 주파수·평활 없이 `rcs_sbr` 를 직접 호출한 값(방위 720점, 회귀 360점). `sigma_smooth`(밴드 5주파수 평균 + 3° 각도창)는 널을 ~18 dB 메워 분포·분위점을 왜곡하므로 통계에 쓰지 않는다(`src/rcs_po.py:191-193`). 문헌 기준값은 원문 PDF 로 확인한 것만 상수로 넣었다(`benchmark/rcs_anchor.py` LITERATURE, 산출 2026-07-24 08:30:28).

**(1) μ(f)=a·f+b 방위 선형평균 회귀** (1.8–6 GHz 0.2 GHz 간격 22점, el=0°). 문헌 기울기와 나란히:

| 기체 (대각 mm) | a [dB/GHz] | b [dBsm] | R²_μ | ε 기울기 c |
|---|---|---|---|---|
| DJI Mini 5 Pro (275) | +1.04 | −25.35 | 0.82 | +0.596 |
| DJI Mavic 4 Pro (441) | +0.96 | −19.10 | 0.80 | +0.170 |
| DJI Matrice 4E (439) | +1.17 | −24.74 | 0.91 | +0.343 |
| DJI Phantom 4 (350) | +1.40 | −23.41 | 0.89 | +0.676 |
| **multiband Phantom 3 (350)** | **0.210** | −19.19 | — | 0.030 |
| **mono3d θ=90° (350)** | **0.315** | −16.15 | — | — |

<sub>기울기 a 는 σ 의 **대역 상승률**이다. 우리 값은 문헌(multiband 0.21·mono3d 0.315 dB/GHz)보다 대체로 **가파르다** — 우리 밴드(1.8–6 GHz)가 소형 표적의 **few-λ 공진영역**이라 σ 가 주파수에 민감하고, 광대역 단일 선형적합이 그 곡률을 한 기울기로 뭉치기 때문이다(R²_μ 도 0.80~0.91 로 낮아 선형성이 약함을 스스로 보고한다). 같은 대각 350 mm 정합기(phantom4)조차 a=+1.40 로 문헌보다 크다 — 절대 기울기는 정량 대조하되 **일치를 주장하지 않는다**.</sub>

**(2) 분포 적합** Rician/Gamma/LogNormal 을 원시 방위 표본에 MLE 적합(3.5 GHz, el=0°). 적합도거리 두 종 — Anderson–Darling(multiband 식5)·Cramér–von Mises(mono3d). ⚠ scipy `rice` 는 **진폭** 분포이고 문헌 Rician 도 통상 진폭 규약이라, 사과-대-사과가 되도록 **진폭 √σ 도메인**에서 적합한다(전력 σ 도메인은 Rician 이 구조적으로 불리):

| 기체 (3.5 GHz · 진폭 √σ) | d_AD Rician | d_CvM Ric / Gam / LogN | 최선 AD·CvM |
|---|---|---|---|
| DJI Mini 5 Pro | 6.518 | 1.080 / 0.647 / 0.529 | LogN·LogN |
| DJI Mavic 4 Pro | 20.515 | 1.954 / 0.524 / 0.373 | LogN·LogN |
| DJI Matrice 4E | 14.408 | 1.637 / 0.556 / 0.603 | Gamma·Gamma |
| DJI Phantom 4 | 23.115 | 2.132 / 0.616 / 0.351 | LogN·LogN |
| **multiband Phantom 3 (AD)** | **0.436** (Rician 최선) | — | Rician |
| **mono3d 평균 (CvM)** | — | **0.28 / 0.31 / 0.48** | Rician<Gamma<LogN |

<sub>진폭 도메인에서 CvM 최선이 **Rician 인 기종은 0/5**, AD 최선이 Rician 인 기종은 0/5 다 — mono3d 가 보고한 **Rician<Gamma<LogNormal** 순위(CvM 0.28/0.31/0.48)와 **정성적으로 일치**한다. 우리 d_AD(Rician)는 multiband 기준 0.436 과 자릿수가 같다. ⚠ 한계: d_CvM 은 노트 식이 √ 를 포함해 **절대치가 √-스케일**이라 **순위 비교만** 안전하다(절대 등치 금지). 또 전력 σ 도메인에서는 변량 불일치로 Gamma/LogNormal 이 이기므로, 분포 선택 결론은 **진폭 도메인 한정**이다.</sub>

**(3) 금속구 절대교정.** 지름 0.5 m(r=0.25, 이론 πr²=−7.07 dBsm)·r=17.8 cm(−10.02 dBsm) PEC 구를 우리 SBR+PO(`group_mat={'metal':'metal'}`)로 돌려 편차를 낸다. 합격선 **< 2 dB** (unified-rcs 실측 편차 1.89 dB):

| 밴드 | 이론 πr² (0.25 / 0.178 m) | 우리 SBR | 편차 [dB] | 합격 <2 dB |
|---|---|---|---|---|
| 1.8 GHz | −7.07 / −10.02 | −7.44 / −10.80 | −0.37 / −0.78 | ✅ |
| 3.5 GHz | −7.07 / −10.02 | −7.05 / −10.09 | +0.02 / −0.07 | ✅ |
| 5.2 GHz | −7.07 / −10.02 | −6.75 / −10.31 | +0.31 / −0.29 | ✅ |
| **unified-rcs 실측 (28 GHz)** | −7.07 (0.25 m) | −8.96 | **1.89** | ✅ (합격선 2) |

<sub>세 밴드·두 반지름 전부 편차 **≤ 0.78 dB** 로 unified-rcs 실측 편차 1.89 dB 보다도 작아 **합격선 2 dB 를 통과**한다. 즉 우리 SBR+PO 는 **해석해가 있는 정준 표적(구)에서는 절대 dBsm 을 <0.1 dB 로 재현**한다 — 절대 스케일 자체는 옳다. 드론 절대값의 불확실성은 교정이 아니라 **few-λ 유전체 형상·편파·자세**에서 온다.</sub>

**(4) el=0° 문헌 대조컷** (원시 방위 선형평균, 5기종×3밴드). 문헌은 전부 **수평면(el=0°)**인데 우리 기본 자세는 el=15° 라, 그 앙각 편향을 el0−el15 로 정량화한다(§2-3 이 2기종만 냈던 것을 5기종으로 확장):

| 기체 | LTE el0−el15 | 5G el0−el15 | WiFi el0−el15 |
|---|---|---|---|
| DJI Mini 5 Pro | +0.79 | +0.43 | +1.81 |
| DJI Mavic 4 Pro | −0.88 | +1.79 | +2.80 |
| DJI Matrice 4E | −0.62 | +1.31 | +2.18 |
| DJI Phantom 4 | +0.60 | +1.17 | +3.50 |

<sub>앙각 편향은 **−0.9 ~ +3.5 dB** 이고 **부호가 밴드·기종마다 다르다**(고역일수록 el=0° 가 밝은 경향). 즉 우리 el=15° 값을 문헌 el=0° 과 견줄 때 밴드에 따라 수 dB 어긋날 수 있으며, 이 표가 그 보정량이다. 문헌과 자세를 맞추려면 el=0° 열을 써야 한다.</sub>

**(5) 평균전력 정규화 분위점 P10/P1** (원시 el=0°, mono3d IV절 지표). 문헌: **−8.5/−18.5 dB (우세성분 有)** · **−10/−20 dB (無, 지수분포 이론 −9.77/−19.98=Swerling I/II)**:

| 기체 | LTE P10/P1 | 5G P10/P1 | WiFi P10/P1 |
|---|---|---|---|
| DJI Mini 5 Pro | −9.71 / −15.19 | −8.11 / −17.08 | −13.01 / −18.34 |
| DJI Mavic 4 Pro | −10.55 / −19.15 | −11.36 / −17.52 | −10.69 / −19.17 |
| DJI Matrice 4E | −8.48 / −12.67 | −10.78 / −19.24 | −10.03 / −18.77 |
| DJI Phantom 4 | −9.78 / −15.74 | −10.44 / −17.33 | −13.31 / −21.34 |

<sub>우리 P1 은 −21.3~−12.7 dB 로 문헌 두 극단(−18.5 / −20) **보다 얕다** — 원시 단일주파수라도 우리 방위 표본이 문헌 실측만큼 깊은 페이딩 꼬리를 만들지 않는다는 뜻이다(우세 정반사 성분이 상대적으로 강함 = Rician 이 이기는 것과 정합). 이 지표를 요동 Swerling 모델의 기종별 선택 근거로 쓸 수 있다(→ Pd 후속).</sub>

---
**추가 선행 방법론 메모 (P1-17·P1-18, 계산 불필요 — md-multiprop 유도).**

- **P1-17 바이스태틱 근사오차 상한.** 산란점을 표적 중심으로 뭉개는 근사의 오차는 `Err ≤ (l²/2)(1/R_T + 1/R_R)` (md-multiprop Appendix D). 우리 챔버 R_T=18.75, R_R=18.61 m, 최대 블레이드 반길이 l≈0.19 m → **Err ≤ 1.9 mm**, 바이스태틱 거리합의 **0.005%** — 표적을 점산란체로 봐도 무방하다.
- **P1-18 거리분해능 이중표기.** 우리 규약은 **바이스태틱 거리합** `ΔR = c/B`; md-multiprop(Willis)은 **이등분선 방향** `ΔR = c/[2B·cos(β/2)]`. 같은 물리인데 축이 다르다(WiFi 예: 3.916 m ÷ (2×0.915) = 2.14 m). ⚠ md-props **학회판**은 `c/(2B)` 라 적었고 **저널판이 정정**했다 — 학회판만 읽고 우리 규약을 '2배 틀렸다' 판정하면 안 된다.

<sub>**§6.1 이 바꾼 것 / 여전히 남는 한계.** 이 정량 대조는 §6 의 *el=0° 대조컷 유예*와 *분포·회귀·구교정 미실시*를 **실제 수치로 대체**한다. 그러나 **절대 dBsm 판정은 여전히 보류**다 — (a) 문헌 앵커 자체가 서로 3.2~3.6 dB(multiband↔mono3d 평균규약)·그 이상(Li&Ling peak<mean) 산포하고, (b) 우리 SBR 은 **스칼라 Γ 라 편파(co/cross)를 분리하지 않는데** 문헌은 co-pol 측정이며, (c) 표적이 문헌은 **Phantom 3**, 우리는 대각만 같은 **Phantom 4**(세대·부품 다름)다. 구 교정이 <0.1 dB 로 통과한다는 것은 **절대 스케일 파이프라인이 옳다**는 뜻이지 드론 절대 σ 가 검증됐다는 뜻이 아니다. 분포 순위(진폭 Rician 우세)·대역 기울기·앙각 편향·분위점은 문헌과 **정성적으로 정합**하되, 절대 점값은 위 세 미정렬 축이 열려 있는 한 판정 보류를 유지한다.</sub>

---
## 정리

1. **밝기는 크기가 정한다.** 가장 큰 기체가 가장 작은 기체보다 **8.2 dB** 밝고, 대역(주파수)은 같은 드론을 **3.2 dB** 밖에 못 움직인다(광학영역). 그리고 그 밝기는 플라스틱 껍데기가 아니라 **속 금속**(모터·배터리·PCB)에서 나온다 — 껍데기는 반투명 스크린일 뿐이다(§2·§3).
2. **프로펠러는 지문을 남긴다.** 돌면 **120~183 Hz** 의 규칙적 깜빡임과 **±1.0~1.6 kHz** 의 날개끝 도플러가 생긴다. 이 지문을 보려면 **가림**이 필수다 — 몸통 뒤 숨은 날개를 세지 않아야 정지 몸통 신호가 부풀지 않고(순수 PO 는 **10~39 dB** 부풀린다), 깜빡임이 그 위로 드러난다(§3·§5).
3. **절대값은 판정 보류, 상대 결론만.** 스톡 Sionna 는 표적 σ 를 못 주므로(→report06) 자작 SBR+PO 로 계산했고(→report07), 그 절대값을 우리 세 밴드를 전부 커버하는 **1급 실측 앵커**(multiband Phantom 3, 방위 **선형평균** — 우리 규약과 동일)와 대각 350 mm 정합기로 견주면 **0.4~2.9 dB 안**에서 맞는다. 다만 문헌 절대앵커 자체가 **12 dB 넘게 산포**하고(Li & Ling aspect-peak 이 다른 실험실 방위-mean 보다도 아래, 물리적으로 불가능) 그 산포가 우리 오차보다 커, **절대 dBsm 은 판정을 보류**한다. 크기 순서·자릿수·대역 추세는 재현되므로 검출은 σ 밴드로 제시해 상대 결론의 robust 함을 보인다(§6).

**이 리포트가 보장하지 않는 것.** 특정 드론의 **절대 dBsm 점값**(문헌 앵커 산포가 우리 오차보다 크다), 플라스틱 셸의 정확한 기여(반투명 불확실 구간), 방위 패턴의 **널 깊이**와 **절대 회전수**(§4·§5 인용 금지), 그리고 아직 정렬 안 된 **앙각 축**(우리 el=15° ↔ 문헌 el=0°, 대조컷은 §6.1 산출완료)과 **편파 축**(스칼라 Γ 라 co-pol/cross-pol 을 분리하지 않는다). 지지하는 것은 상대 순서·대역 추세·자세 구조다.

> **다음 리포트**: [report09](report09.ipynb) — 이제 **탐지**로 넘어간다. 그 전에 챔버 **바닥이 놓는 함정**(표적을 경유해 되돌아오는 유령 신호)을 먼저 본다.